In [1]:
import sys
sys.path.insert(0, "../../../")

import numpy as np
import astropy.units as u
import matplotlib.pyplot as plt

from agnpy.emission_regions import Blob
from agnpy.targets import RingDustTorus
from agnpy.spectra import ExpCutoffPowerLaw

from astropy.constants import m_p
from agnpy.utils.conversion import mec2, mpc2
from agnpy.utils.math import axes_reshaper, log10
from agnpy.photo_meson.kernels import eta_0, secondaries, interpolate_g_parameter, gKernel

In [2]:
L_disk = 2 * 1e46 * u.Unit("erg s-1")
T_dt = 1e3 * u.K
xi_dt = 0.1

dust_torus = RingDustTorus(L_disk, xi_dt, T_dt)

print("u = ",dust_torus.u(0.*u.Unit("cm")))
print("epsilion = ", dust_torus.epsilon_dt*mec2)
print(dust_torus.R_dt)

u =  2.1668724321943448e-05 erg / cm3
epsilion =  3.7277523000000007e-13 erg
1.565247584249853e+19 cm


In [3]:
factor = 1e0
E_star = 3e20 * u.Unit("eV")
gamma_star = (E_star / mpc2).to_value("")

Gamma = 1500
theta_s = 120*u.deg
R_b = 1e16*u.Unit("cm")

blob = Blob(R_b=R_b)
blob.set_delta_D(Gamma=Gamma, theta_s=theta_s)

delta_D = blob.delta_D

n_p = ExpCutoffPowerLaw.from_total_energy_density(
    1.0 * u.Unit("erg/cm3"),
    mass = m_p,
    p = 2,
    gamma_c = factor * gamma_star / delta_D,
    gamma_min = (1.0 * u.Unit("GeV") / (mpc2 * delta_D)).to_value(""),
    gamma_max = 30.0 * factor * gamma_star / delta_D
)

blob.n_p = n_p

print(delta_D)

0.00044444447736626127


In [4]:
prefactor = blob.V_b / mpc2

def N_p_prim(gamma_prim):
    return prefactor * n_p(gamma_prim)

In [5]:
def f_ph(r):
    return dust_torus.u(r)/(2.*np.pi*dust_torus.epsilon_dt*mec2)

f_ph(0.*u.Unit("cm"))

<Quantity 9251378.06590013 1 / cm3>

In [6]:
def E_p_prim(E_p):
    return E_p/delta_D

print(E_p_prim(1.*u.Unit("erg")))

2249.9998333333147 erg


In [7]:
def gamma_p_prim(gamma_p):
    return gamma_p/delta_D

print(gamma_p_prim(1.))

2249.9998333333147


In [8]:
def theta_pgam(phi,theta_s,r,R):
    rR = r/R
    sqr = np.sqrt(1+rR**2)
    return np.acos(rR*np.cos(theta_s)/sqr + np.sin(theta_s)*np.cos(np.pi/2+phi)/sqr) # in rad

# r = 0. * u.Unit("cm")
# R = 100 * u.Unit("cm")
# phi = 90. * np.pi/180. # in rad
# theta_s = 10. * np.pi/180. # in rad
# print(theta_pgam(phi,theta_s,r,R)*180./np.pi)

In [9]:
class PhotoMesonProduction:

    def __init__(self, blob, target, r, theta_s, particle, integrator=np.trapz):
        self.blob = blob
        # check that this blob has a proton distribution
        if self.blob._n_p is None:
            raise AttributeError(
                "There is no proton distribution in this emission region"
            )
        self.target = target
        self.integrator = integrator
        self.theta_s = theta_s
        self.r = r
        self.particle = particle

        return

    def H(self, phi, E, r_b, g_kernel, integrator=np.trapz):

        # Integral on E_p to be made from E to infinity
        _phi, _E = axes_reshaper(phi, E)  # shape (len(eta), 1), (1, len(E))

        _E_min = _E.copy()
        _E_max = _E.copy() * 1e8

        Emin = self.blob.n_p.gamma_min * mpc2
        Emax = self.blob.n_p.gamma_max * mpc2

        _E_min = np.clip(_E_min, Emin, None)  # replace values smaller than Emin
        _E_max = np.clip(_E_max, None, Emax)  # replace values larger than Emax

        _E_p = np.logspace(
            log10(_E_min.to_value("eV")), log10(_E_max.to_value("eV")), 200
        ) * u.Unit(
            "eV"
        )  # shape (200, 1, len(E))

        _gamma_p_prim = _E_p / (mpc2 * delta_D)
        _x = _E / _E_p
        _eta = 4.*_E_p * self.target.epsilon_dt*mec2 / mpc2**2
        _eta = _eta.to("")

        _H_integrand = (
            1/_E_p
            * f_ph(self.r)
            * delta_D**3*N_p_prim(_gamma_p_prim)/(4.*np.pi)
            
            # =================================================================================================================================================================
            # =============================================================================================================================== Add description of the units!!! =
            # =================================================================================================================================================================

            * g_kernel(_eta*eta_0,theta_pgam(_phi,self.theta_s,r_b,self.target.R_dt)*180/np.pi,_x) #!!!!! gKernel angle in degrees!!! See tables!!! -> change to rad!!!!
        ).to("erg-2 cm-3")

        _H = integrator(
            _H_integrand,
            _E_p,
            axis=0,
        ).to("erg-1 cm-3")
        return _H

    def evaluate_spectrum(self, E, r_b, particle, integrator=np.trapz):
  
        if particle not in secondaries:
            raise AttributeError(
                f"There is no secondary particle from photomeson interactions named {particle}."
            )

        g_kernel = gKernel(self.particle)
        # Integral on phi angle to be done from 0 to 2 pi
        phi = np.linspace(
            0.,
            2*np.pi,
            100,
        )
        _H = self.H(
            phi,
            E,
            r_b,
            g_kernel,
            integrator=integrator,
        )
        dN_dEdtdOmega = integrator(_H, phi, axis=0).to("erg-1 cm-3")
        return dN_dEdtdOmega


In [10]:
E = np.logspace(10, 20, 10) * u.Unit("eV")
r_b = 1e16*u.Unit("cm")

kernel = PhotoMesonProduction(blob=blob,target=dust_torus,r = 1*u.Unit("cm"),theta_s=0.,particle="gamma")
kernel.evaluate_spectrum(E,r_b,"gamma")

/home/piotr/Dokumenty/CODE/agnpy_cosimo/.venv/lib/python3.12/site-packages/astropy/units/quantity.py:648: RuntimeWarning: divide by zero encountered in log10
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/piotr/Dokumenty/CODE/agnpy_cosimo/.venv/lib/python3.12/site-packages/astropy/units/quantity.py:648: RuntimeWarning: divide by zero encountered in power
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/piotr/Dokumenty/CODE/agnpy_cosimo/.venv/lib/python3.12/site-packages/astropy/units/quantity.py:648: RuntimeWarning: invalid value encountered in multiply
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/tmp/ipykernel_186100/4027268399.py:55: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  _H = integrator(
/tmp/ipykernel_186100/4027268399.py:83: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or on

<Quantity [       0.        , 10646883.81960925, 12918069.53858752,
           15677585.39160702, 18969017.82877742, 19968399.53111023,
           20164117.07567659,  7885466.85526601,    24159.89097723,
                  0.        ] 1 / (erg cm3)>

In [11]:
AA = gKernel("gamma")
AA(10.1*eta_0,np.pi/2 * 180/np.pi,1e-3)

array(9.48586573e-17)

In [12]:
from agnpy.photo_meson.photo_meson import PhotoMesonProductionAngular

In [13]:
kernel = PhotoMesonProductionAngular(blob=blob,target=dust_torus,r = 1*u.Unit("cm"),theta_s=0.,particle="gamma")
kernel.evaluate_spectrum(E,r_b,"gamma")

/home/piotr/Dokumenty/CODE/agnpy_cosimo/.venv/lib/python3.12/site-packages/astropy/units/quantity.py:648: RuntimeWarning: divide by zero encountered in log10
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/piotr/Dokumenty/CODE/agnpy_cosimo/.venv/lib/python3.12/site-packages/astropy/units/quantity.py:648: RuntimeWarning: divide by zero encountered in power
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/home/piotr/Dokumenty/CODE/agnpy_cosimo/.venv/lib/python3.12/site-packages/astropy/units/quantity.py:648: RuntimeWarning: invalid value encountered in multiply
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


<Quantity [       0.        , 10646883.81960925, 12918069.53858752,
           15677585.39160702, 18969017.82877742, 19968399.53111023,
           20164117.07567659,  7885466.85526601,    24159.89097723,
                  0.        ] 1 / (erg cm3)>